In [9]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
import plotly.graph_objects as go

In [41]:
# ==============================================================================
# 1. PARÁMETROS CONFIGURABLES DEL PROYECTO Y DEL DCF
# ==============================================================================
N_SIMS = 20000  # Número de simulaciones de Monte Carlo
N_YEARS = 1  # Horizonte del DCF en años (n años hacia el futuro)
DT_SIM = 1/252  # Paso de tiempo de simulación en años (1.0 = anual, 1/12 = mensual)
WACC = 0.10  # Tasa de descuento r = 10% (0.10)

# Volúmenes anuales de producción/venta (ajusta según el tamaño de tu empresa)
OIL_PROD_ANNUAL = 1_000_000  # Barriles de petróleo vendidos por año (bbl/año)
GAS_PROD_ANNUAL = 5_000_000  # Millones de BTU de gas vendidos por año (mmBTU/año)

# Parámetros de descarga histórica para calibración
HIST_START = '2005-01-01'  # 20+ años de historia mensual recomendada para DCF
HIST_END = '2026-11-01'

In [43]:
# ==============================================================================
# 2. DESCARGA DE DATOS HISTÓRICOS REALES (WTI OIL & HENRY HUB GAS)
# ==============================================================================
print('Descargando datos históricos mensuales desde Yahoo Finance...')
tickers = ['CL=F', 'NG=F']  # CL=F: Petróleo WTI | NG=F: Gas Natural Henry Hub
raw_data = yf.download(
    tickers, start=HIST_START, end=HIST_END, interval='1mo', progress=False
)['Close']

# Limpiar y alinear fechas
df_prices = raw_data.rename(columns={'CL=F': 'Oil', 'NG=F': 'Gas'}).dropna()
print(
    f'Datos descargados: {len(df_prices)} observaciones mensuales ({df_prices.index[0].strftime("%Y-%m")} a {df_prices.index[-1].strftime("%Y-%m")})'
)

Descargando datos históricos mensuales desde Yahoo Finance...
Datos descargados: 224 observaciones mensuales (2005-01 a 2026-10)


In [44]:
# ==============================================================================
# 3. CALIBRACIÓN DE PARÁMETROS OU POR OLS (AR(1) EN LOG-PRECIOS)
# ==============================================================================
def calibrate_ou_ols(price_series, dt_hist=1 / 12):
    """
    Calibra un proceso Ornstein-Uhlenbeck (OU) sobre el logaritmo natural del precio:
    dX_t = theta * (mu - X_t) dt + sigma * dW_t, donde X_t = ln(P_t)
    """
    X = np.log(price_series.values)
    X_t = X[1:]
    X_prev = sm.add_constant(X[:-1])

    # Regresión OLS: X_t = a + b * X_{t-1} + eps_t
    model = sm.OLS(X_t, X_prev).fit()
    a, b = model.params
    resid = model.resid
    sd_eps = np.std(resid, ddof=2)

    # Validar reversión a la media (0 < b < 1)
    if not (0 < b < 1):
        print(
            f'Advertencia: b={b:.4f} fuera de (0,1). La serie muestra tendencia o raíz unitaria.'
        )

    # Conversión exacta de AR(1) discreto a parámetros continuos OU
    theta = -np.log(b) / dt_hist
    mu = a / (1 - b)
    sigma = sd_eps * np.sqrt((-2 * np.log(b)) / (dt_hist * (1 - b**2)))

    # Precio de equilibrio de largo plazo en dólares (con ajuste de Jensen)
    eq_price = np.exp(mu + (sigma**2) / (4 * theta))

    return {
        'theta': theta,
        'mu': mu,
        'sigma': sigma,
        'eq_price': eq_price,
        'last_price': price_series.iloc[-1],
        'resid': resid,
    }

In [45]:
oil_params = calibrate_ou_ols(df_prices['Oil'], dt_hist=1 / 12)
gas_params = calibrate_ou_ols(df_prices['Gas'], dt_hist=1 / 12)

# Calcular correlación histórica entre los choques (residuos) de Petróleo y Gas
rho = np.corrcoef(oil_params['resid'], gas_params['resid'])[0, 1]

print('\n--- PARÁMETROS CALIBRADOS (PROCESO OU LOGARÍTMICO) ---')
print(
    f"Petróleo (WTI) -> Actual: ${oil_params['last_price']:.2f} | Equilibrio LP: ${oil_params['eq_price']:.2f} | theta: {oil_params['theta']:.4f} | sigma: {oil_params['sigma']:.2%}"
)
print(
    f"Gas Natural    -> Actual: ${gas_params['last_price']:.2f} | Equilibrio LP: ${gas_params['eq_price']:.2f} | theta: {gas_params['theta']:.4f} | sigma: {gas_params['sigma']:.2%}"
)
print(f'Correlación entre choques de Petróleo y Gas (rho): {rho:.4f}')


--- PARÁMETROS CALIBRADOS (PROCESO OU LOGARÍTMICO) ---
Petróleo (WTI) -> Actual: $104.12 | Equilibrio LP: $75.26 | theta: 1.1372 | sigma: 45.04%
Gas Natural    -> Actual: $2.90 | Equilibrio LP: $4.05 | theta: 0.8449 | sigma: 58.26%
Correlación entre choques de Petróleo y Gas (rho): 0.0114


In [46]:
# ==============================================================================
# 4. SIMULACIÓN DE MONTE CARLO (CAMINOS ALEATORIOS CORRELACIONADOS)
# ==============================================================================
steps_per_year = int(round(1 / DT_SIM))
total_steps = N_YEARS * steps_per_year

np.random.seed(42)

# Matrices de log-precios X_t = ln(P_t): filas = pasos de tiempo, columnas = simulaciones
X_oil = np.zeros((total_steps + 1, N_SIMS))
X_gas = np.zeros((total_steps + 1, N_SIMS))

X_oil[0] = np.log(oil_params['last_price'])
X_gas[0] = np.log(gas_params['last_price'])

# Constantes de la solución exacta discretizada para el paso DT_SIM elegido
exp_oil = np.exp(-oil_params['theta'] * DT_SIM)
std_oil = oil_params['sigma'] * np.sqrt(
    (1 - np.exp(-2 * oil_params['theta'] * DT_SIM)) / (2 * oil_params['theta'])
)

exp_gas = np.exp(-gas_params['theta'] * DT_SIM)
std_gas = gas_params['sigma'] * np.sqrt(
    (1 - np.exp(-2 * gas_params['theta'] * DT_SIM)) / (2 * gas_params['theta'])
)

for t in range(1, total_steps + 1):
    # Ruido normal estándar independiente N(0,1)
    Z1 = np.random.standard_normal(N_SIMS)
    Z2 = np.random.standard_normal(N_SIMS)

    # Descomposición de Cholesky para correlacionar el ruido de Gas y Petróleo
    Z_oil = Z1
    Z_gas = rho * Z1 + np.sqrt(1 - rho**2) * Z2

    # Evolución exacta del proceso OU en log-precios
    X_oil[t] = (
        X_oil[t - 1] * exp_oil + oil_params['mu'] * (1 - exp_oil) + std_oil * Z_oil
    )
    X_gas[t] = (
        X_gas[t - 1] * exp_gas + gas_params['mu'] * (1 - exp_gas) + std_gas * Z_gas
    )

# Convertir de vuelta a precios en dólares P_t = exp(X_t)
P_oil_paths = np.exp(X_oil)
P_gas_paths = np.exp(X_gas)

# Si DT_SIM < 1 (ej. mensual), promediamos cada año para obtener precios anuales del DCF
# Si DT_SIM == 1, toma directamente los precios de cada año 1..N_YEARS
P_oil_annual = P_oil_paths[1:].reshape(N_YEARS, steps_per_year, N_SIMS).mean(axis=1)
P_gas_annual = P_gas_paths[1:].reshape(N_YEARS, steps_per_year, N_SIMS).mean(axis=1)

In [47]:
# ==============================================================================
# 5. CÁLCULO DEL DCF (FLUJO DE CAJA DESCONTADO) Y DISTRIBUCIÓN DEL NPV
# ==============================================================================
# Flujos de caja anuales (Ingresos por venta de Oil + Gas, sin otros costos por ahora)
FCF_annual = (P_oil_annual * OIL_PROD_ANNUAL) + (P_gas_annual * GAS_PROD_ANNUAL)

# Vector de factores de descuento: 1 / (1 + r)^t para t = 1, 2, ..., N_YEARS
years_vec = np.arange(1, N_YEARS + 1).reshape(-1, 1)
discount_factors = 1 / ((1 + WACC) ** years_vec)

# Valor Presente Neto (NPV) para cada una de las N_SIMS trayectorias (en Millones de $)
NPV_distribution = (FCF_annual * discount_factors).sum(axis=0) / 1e6

# Estadísticas clave del NPV
mean_npv = np.mean(NPV_distribution)
median_npv = np.median(NPV_distribution)
p10_npv = np.percentile(NPV_distribution, 10)
p90_npv = np.percentile(NPV_distribution, 90)
std_npv = np.std(NPV_distribution)

print(f'\n--- RESULTADOS DEL DCF ({N_SIMS:,} SIMULACIONES A {N_YEARS} AÑOS, r={WACC:.0%}) ---')
print(f'NPV Esperado (Media):   ${mean_npv:,.2f} M')
print(f'NPV Mediano (P50):      ${median_npv:,.2f} M')
print(f'Escenario Bajo (P10):   ${p10_npv:,.2f} M')
print(f'Escenario Alto (P90):   ${p90_npv:,.2f} M')
print(f'Desviación Estándar:    ${std_npv:,.2f} M')


--- RESULTADOS DEL DCF (20,000 SIMULACIONES A 1 AÑOS, r=10%) ---
NPV Esperado (Media):   $98.93 M
NPV Mediano (P50):      $97.56 M
Escenario Bajo (P10):   $80.59 M
Escenario Alto (P90):   $119.20 M
Desviación Estándar:    $15.28 M


In [48]:
# ==============================================================================
# 6. VISUALIZACIÓN INTERACTIVA EN PLOTLY (3 GRÁFICAS SEPARADAS)
# ==============================================================================
time_grid = np.linspace(0, N_YEARS, total_steps + 1)
n_paths_plot = 80  # Cantidad de trayectorias individuales a dibujar

In [49]:
# ------------------------------------------------------------------------------
# GRÁFICA 1: Trayectorias de Monte Carlo - Petróleo WTI
# ------------------------------------------------------------------------------
fig_oil = go.Figure()

# Caminos individuales (en gris/azul translúcido para no saturar)
for i in range(n_paths_plot):
    fig_oil.add_trace(
        go.Scatter(
            x=time_grid,
            y=P_oil_paths[:, i],
            mode='lines',
            line=dict(color='rgba(43, 92, 143, 0.20)', width=1),
            showlegend=(i == 0),
            name='Caminos aleatorios (muestra)',
            hoverinfo='skip',
        )
    )

# Media de las N_SIMS simulaciones
fig_oil.add_trace(
    go.Scatter(
        x=time_grid,
        y=P_oil_paths.mean(axis=1),
        mode='lines',
        line=dict(color='black', width=3),
        name=f'Media ({N_SIMS:,} simulaciones)',
    )
)

# Nivel de equilibrio de largo plazo
fig_oil.add_hline(
    y=oil_params['eq_price'],
    line_dash='dash',
    line_color='red',
    line_width=2,
    annotation_text=f"Equilibrio LP: ${oil_params['eq_price']:.2f}/bbl",
    annotation_position='top right',
)

fig_oil.update_layout(
    title=f'Simulación Monte Carlo: Precio del Petróleo WTI ({N_YEARS} años)',
    xaxis_title='Años',
    yaxis_title='Precio ($/bbl)',
    template='plotly_dark',
    width=950,
    height=500,
    hovermode='x unified',
)
fig_oil.show()

In [50]:
# ------------------------------------------------------------------------------
# GRÁFICA 2: Trayectorias de Monte Carlo - Gas Natural Henry Hub
# ------------------------------------------------------------------------------
fig_gas = go.Figure()

for i in range(n_paths_plot):
    fig_gas.add_trace(
        go.Scatter(
            x=time_grid,
            y=P_gas_paths[:, i],
            mode='lines',
            line=dict(color='rgba(0, 230, 118, 0.22)', width=1),
            showlegend=(i == 0),
            name='Caminos aleatorios (muestra)',
            hoverinfo='skip',
        )
    )

fig_gas.add_trace(
    go.Scatter(
        x=time_grid,
        y=P_gas_paths.mean(axis=1),
        mode='lines',
        line=dict(color='#ffffff', width=3),
        name=f'Media ({N_SIMS:,} simulaciones)',
    )
)

fig_gas.add_hline(
    y=gas_params['eq_price'],
    line_dash='dash',
    line_color='#ff5252',
    line_width=2,
    annotation_text=f"Equilibrio LP: ${gas_params['eq_price']:.2f}/mmBTU",
    annotation_position='top right',
    annotation_font_color='#ff5252',
)

fig_gas.update_layout(
    title=f'Simulación Monte Carlo: Precio del Gas Natural Henry Hub ({N_YEARS} años)',
    xaxis_title='Años',
    yaxis_title='Precio ($/mmBTU)',
    template='plotly_dark',
    width=950,
    height=500,
    hovermode='x unified',
)
fig_gas.show()

In [51]:
# ------------------------------------------------------------------------------
# GRÁFICA 3: Distribución de Frecuencias del NPV (DCF)
# ------------------------------------------------------------------------------
fig_npv = go.Figure()

fig_npv.add_trace(
    go.Histogram(
        x=NPV_distribution,
        nbinsx=70,
        marker_color='#3b82f6',
        marker_line_color='#111111',
        marker_line_width=0.8,
        opacity=0.88,
        name='Frecuencia NPV',
    )
)

# Líneas verticales para Media, P10 y P90
fig_npv.add_vline(
    x=mean_npv,
    line_dash='solid',
    line_color='#ff5252',
    line_width=2.5,
    annotation_text=f'Media: ${mean_npv:,.1f}M',
    annotation_position='top right',
    annotation_font_color='#ff5252',
)

fig_npv.add_vline(
    x=p10_npv,
    line_dash='dash',
    line_color='#ffb74d',
    line_width=2,
    annotation_text=f'P10: ${p10_npv:,.1f}M',
    annotation_position='top left',
    annotation_font_color='#ffb74d',
)

fig_npv.add_vline(
    x=p90_npv,
    line_dash='dash',
    line_color='#69f0ae',
    line_width=2,
    annotation_text=f'P90: ${p90_npv:,.1f}M',
    annotation_position='top right',
    annotation_font_color='#69f0ae',
)

fig_npv.update_layout(
    title=f'Distribución del Valor Presente Neto (NPV) — {N_SIMS:,} Simulaciones a {N_YEARS} Años (r = {WACC:.0%})',
    xaxis_title='NPV (Millones de USD)',
    yaxis_title='Frecuencia (N° de escenarios)',
    template='plotly_dark',
    width=950,
    height=500,
    bargap=0.05,
    showlegend=False,
)
fig_npv.show()